# Full Reaction Network

\begin{align}
CO_2 + 3H_2 &\rightleftharpoons CH_3OH + H_2O\\
CO + 2H_2 &\rightleftharpoons CH_3OH\\
CO_2 + H_2 &\rightleftharpoons CO + H_2O\\
2CH_3OH &\rightleftharpoons CH_3OCH_3 + H_2O
\end{align}


# Yield

"How much of Educt i is converted into Product k"

\begin{equation}
    Y_{k} = \frac{\nu_i}{\nu_k} \, \frac{n_{k, 0}-n_{k}}{n_{i,0}}
\end{equation}

Educt is Methanol, everything else is a byproduct

# Reaction extend

\begin{equation}
    \xi_{i} = \frac{n_i - n_{i,0}}{\nu_i}
\end{equation}

# Importing the researched Shomate Data-Sets

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as scp
import pandas as pd

# Pfad zu den Daten
data_path = r'c:\Users\Peter\Documents\Uni\Master\CRE 3\CRE3-Assignment-2\data_literature\NASA_shomate_coefficients_full.csv'

# NASA Shomate Koeffizienten laden
df_nasa = pd.read_csv(data_path)
print("\nGeladene Spezies:")
print(df_nasa[['species_name', 'formula', 'T1_min_K', 'T1_max_K', 'T2_min_K', 'T2_max_K']].to_string())


Geladene Spezies:
  species_name  formula  T1_min_K  T1_max_K  T2_min_K  T2_max_K
0           H2       H2     200.0    1000.0    1000.0    6000.0
1          H2O      H2O     200.0    1000.0    1000.0    6000.0
2           CO       CO     200.0    1000.0    1000.0    6000.0
3          CO2      CO2     200.0    1000.0    1000.0    6000.0
4        CH3OH    CH3OH     200.0    1000.0    1000.0    6000.0
5      CH3OCH3  CH3OCH3     200.0    1000.0    1000.0    6000.0


# Formulating the Matrix of stoichometric coefficients

\begin{equation}
  \underline{N}^T=
    \begin{bmatrix}
    & CO & CO_2 & CH_3OCH_3 & H_2 & H_2O & CH_3OH \\
R_1 & 0 & -1 & 0 & -3 & 1 & 1 \\
R_2 & -1 & 0 & 0 & -2 & 0 & 1 \\
R_3 & 1 & -1 & 0 & -1 & 1 & 0 \\
R_4 & 0 & 0 & 1 & 0 & 1 & -2
    \end{bmatrix}
\end{equation}

To ensure, that the mathematical method of the matrix of stochiometric coefficients is possible, we need to snure, that $N_{1,1}$ has an inverse matrix that can be calculated. A quick calculation shows, that R3 is equal to R1 -R2 ($R_1 - R_2 = R_3$). 

Therefor, R4 and R3 are interchanged, to enable a succesfull solution of the mathematical approach:

\begin{equation}
  \underline{N}^T=
    \begin{bmatrix}
    & CO & CO_2 & CH_3OCH_3 & H_2 & H_2O & CH_3OH \\
R_1 & 0 & -1 & 0 & -3 & 1 & 1 \\
R_2 & -1 & 0 & 0 & -2 & 0 & 1 \\
R_4 & 0 & 0 & 1 & 0 & 1 & -2  \\
R_3 & 1 & -1 & 0 & -1 & 1 & 0 
    \end{bmatrix}
\end{equation}

In [14]:
import numpy as np
from numpy.linalg import matrix_rank

# --- DME Synthesis Stoichiometric Analysis ---
# Component Order: [CO, CO2, CH3OCH3 (DME), H2, H2O, CH3OH]
# Reaction Order adjusted: [R1, R2, R4, R3] to ensure N11 is non-singular
# This order ensures Key Components are in the first 3 rows (Rank = 3)

n_matrix = np.array([
    [ 0, -1,  0,  1],  # CO    
    [-1,  0,  0, -1],  # CO2   
    [ 0,  0,  1,  0],  # CH3OCH3 (DME) -> Now independent in the 3rd column (because R1-R2=R3)
    [-3, -2,  0, -1],  # H2    
    [ 1,  0,  1,  1],  # H2O   
    [ 1,  1, -2,  0]   # CH3OH
])

# Get the transposed matrix (N^T)
n_transposed = n_matrix.T

# Determine the rank of the matrix
n_rank = matrix_rank(n_matrix)

print("--- DME Synthesis Stoichiometric Analysis ---")
print(f"Sequence: [CO, CO2, CH3OCH3 (DME), H2, H2O, CH3OH]")
print("\nTransposed Matrix (N^T):")
print(n_transposed)
print(f"\nMatrix Rank (Number of Key Reactions): {n_rank}")

--- DME Synthesis Stoichiometric Analysis ---
Sequence: [CO, CO2, CH3OCH3 (DME), H2, H2O, CH3OH]

Transposed Matrix (N^T):
[[ 0 -1  0 -3  1  1]
 [-1  0  0 -2  0  1]
 [ 0  0  1  0  1 -2]
 [ 1 -1  0 -1  1  0]]

Matrix Rank (Number of Key Reactions): 3


The rank is $R_N=3$, which means that three key reactions and components are sufficient to describe the reaction extent based on stoichiometry. According to the order of components and reactions chosen for the matrix of stoichiometric coefficients, $CO$, $CO_2$, $CH_3OCH_3 (DME)$ are the key components. 

## 2. Thermodynamische Funktionen (Shomate-Gleichung NASA-Format)

NASA-Shomate Koeffizienten (7 pro Bereich):
$$C_p = a_1 + a_2 T + a_3 T^2 + a_4 T^3 + \frac{a_5}{T^2}$$
$$H = a_1 T + \frac{a_2 T^2}{2} + \frac{a_3 T^3}{3} + \frac{a_4 T^4}{4} - \frac{a_5}{T} + a_6$$
$$S = a_1 \ln(T) + a_2 T + \frac{a_3 T^2}{2} + \frac{a_4 T^3}{3} - \frac{a_5}{2T^2} + a_7$$

In [13]:
# Storing Shomate-coefficients in Dictionary
SHOMATE_NASA = {}

for _, row in df_nasa.iterrows():
    species = row['species_name'].strip()
    # Checks, if the given row has only one or two temperature ranges. If T2_min_K is NaN, there is only one range.
    if pd.isna(row['T2_min_K']): 
        # Only one Temperature range
        ranges = [(
            row['T1_min_K'], row['T1_max_K'],
            row['a1_T1'], row['a2_T1'], row['a3_T1'], row['a4_T1'], row['a5_T1'],
            row['a6_T1'], row['a7_T1'], row['b1_T1'], row['b2_T1']
        )]
    else:
        # Two temperature ranges
        ranges = [
            (
                row['T1_min_K'], row['T1_max_K'],
                row['a1_T1'], row['a2_T1'], row['a3_T1'], row['a4_T1'], row['a5_T1'],
                row['a6_T1'], row['a7_T1'], row['b1_T1'], row['b2_T1']
            )
        ]
        # Additional check to ensure that the second range is valid before appending
        if not pd.isna(row['T2_min_K']):
            ranges.append((
                row['T2_min_K'], row['T2_max_K'],
                row['a1_T2'], row['a2_T2'], row['a3_T2'], row['a4_T2'], row['a5_T2'],
                row['a6_T2'], row['a7_T2'], row['b1_T2'], row['b2_T2']
            ))
    SHOMATE_NASA[species] = ranges

for sp, ranges in SHOMATE_NASA.items():
    print(f"  {sp}: {len(ranges)} range(s)")
    for r in ranges:
        print(f"    {r[0]:.0f} – {r[1]:.0f} K")

  H2: 2 range(s)
    200 – 1000 K
    1000 – 6000 K
  H2O: 2 range(s)
    200 – 1000 K
    1000 – 6000 K
  CO: 2 range(s)
    200 – 1000 K
    1000 – 6000 K
  CO2: 2 range(s)
    200 – 1000 K
    1000 – 6000 K
  CH3OH: 2 range(s)
    200 – 1000 K
    1000 – 6000 K
  CH3OCH3: 2 range(s)
    200 – 1000 K
    1000 – 6000 K
